# Return Calculations 101

**Level:** Beginner  
**Tags:** Returns, Math, Visualization

## Overview

Compute simple, log, and cumulative returns with inline checks and visuals

## References

1. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson.
2. Shreve, S.E. (2004). *Stochastic Calculus for Finance II*. Springer.
3. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
np.random.seed(42)

print('Libraries imported successfully!')

## 1. Introduction to Return Calculations

### Why Returns Matter

In finance, we rarely care about absolute price levels. Instead, we focus on **returns** - the percentage change in value over time. Returns allow us to:

- **Compare investments** of different sizes and prices
- **Aggregate performance** across time periods
- **Measure risk** through return volatility
- **Build predictive models** for future performance

### Types of Returns

We'll cover three main types of returns:

1. **Simple Returns (Arithmetic Returns)** - Intuitive percentage changes
2. **Logarithmic Returns (Log Returns)** - Mathematically convenient for modeling
3. **Cumulative Returns** - Total return over multiple periods

### Learning Objectives

By the end of this notebook, you will:

✓ Understand the difference between simple and log returns  
✓ Calculate returns from price data using Python  
✓ Visualize return distributions and time series  
✓ Understand when to use each type of return  
✓ Apply returns to real-world portfolio analysis  

### Real-World Context

Returns are the foundation of:
- **Portfolio management** - Measuring manager performance
- **Risk management** - Calculating VaR, volatility
- **Trading strategies** - Backtesting and optimization
- **Derivatives pricing** - Modeling stock price dynamics
- **Research** - Academic finance studies

Let's dive in!

In [ ]:
# Generate sample price data for demonstration
# Simulating daily stock prices for one year (252 trading days)

# Set parameters
initial_price = 100
n_days = 252
daily_drift = 0.0005  # 0.05% average daily return
daily_volatility = 0.02  # 2% daily volatility

# Generate random returns using normal distribution
np.random.seed(42)
random_returns = np.random.normal(daily_drift, daily_volatility, n_days)

# Convert to price series (geometric random walk)
prices = initial_price * np.exp(np.cumsum(random_returns))

# Create date range
dates = pd.date_range(start='2024-01-01', periods=n_days, freq='B')  # B = business days

# Create DataFrame
df_prices = pd.DataFrame({
    'Date': dates,
    'Price': prices
})
df_prices.set_index('Date', inplace=True)

# Display first and last few rows
print("Sample Stock Price Data")
print("=" * 60)
print(f"\nFirst 5 days:")
print(df_prices.head())
print(f"\nLast 5 days:")
print(df_prices.tail())
print(f"\nPrice Statistics:")
print(f"  Starting Price: ${df_prices['Price'].iloc[0]:.2f}")
print(f"  Ending Price: ${df_prices['Price'].iloc[-1]:.2f}")
print(f"  Minimum Price: ${df_prices['Price'].min():.2f}")
print(f"  Maximum Price: ${df_prices['Price'].max():.2f}")
print(f"  Total Return: {(df_prices['Price'].iloc[-1] / df_prices['Price'].iloc[0] - 1) * 100:.2f}%")

# Quick visualization
plt.figure(figsize=(14, 5))
plt.plot(df_prices.index, df_prices['Price'], linewidth=2, color='steelblue')
plt.fill_between(df_prices.index, df_prices['Price'], alpha=0.3, color='steelblue')
plt.title('Simulated Stock Price - One Year of Daily Data', fontsize=14, fontweight='bold')
plt.xlabel('Date', fontsize=11)
plt.ylabel('Price ($)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Mathematical Foundation

### 2.1 Simple Returns (Arithmetic Returns)

**Definition:** The simple return measures the percentage change in price from one period to the next.

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

Where:
- $R_t$ = simple return at time $t$
- $P_t$ = price at time $t$
- $P_{t-1}$ = price at time $t-1$

**Properties:**
- ✓ Intuitive and easy to understand
- ✓ Bounded below by -100% (can't lose more than everything)
- ✗ Not time-additive: $(1+R_1)(1+R_2) \neq 1 + R_1 + R_2$
- ✗ Not symmetric: +100% then -50% ≠ 0% total return

**Example:**
- Stock price increases from \$100 to \$110
- Simple return = $(110 - 100) / 100 = 0.10 = 10\%$

---

### 2.2 Logarithmic Returns (Log Returns)

**Definition:** The logarithmic return is the natural log of the price ratio.

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})$$

Alternatively:
$$r_t = \ln(1 + R_t)$$

**Properties:**
- ✓ **Time-additive:** $r_{total} = r_1 + r_2 + \cdots + r_n$
- ✓ **Symmetric:** $\ln(1.5) = 0.405$, $\ln(0.667) = -0.405$
- ✓ **Statistically convenient** - closer to normal distribution
- ✓ **Continuous compounding** framework
- ✗ Less intuitive than simple returns

**Why Log Returns?**

1. **Time Aggregation is Simple Addition:**
   - If Day 1 return = 2% and Day 2 return = 3%
   - Total log return = 2% + 3% = 5% ✓
   - Total simple return ≠ 2% + 3% (it's 5.06%) ✗

2. **Mathematical Modeling:**
   - Stock prices modeled as Geometric Brownian Motion use log returns
   - $dS_t = \mu S_t dt + \sigma S_t dW_t$
   - This implies log returns are normally distributed

3. **Symmetry:**
   - Going from \$100 to \$150 (+50%) then back to \$100 (-33.3%)
   - Log returns: +40.5% and -40.5% (symmetric!)
   - Simple returns: +50% and -33.3% (asymmetric)

---

### 2.3 Relationship Between Simple and Log Returns

For small returns (|R| < 10%), the two are approximately equal:

$$r_t \approx R_t \quad \text{when } R_t \text{ is small}$$

More precisely, using Taylor expansion:
$$\ln(1 + R_t) = R_t - \frac{R_t^2}{2} + \frac{R_t^3}{3} - \cdots$$

**Conversion formulas:**
$$r_t = \ln(1 + R_t)$$
$$R_t = e^{r_t} - 1$$

---

### 2.4 Multi-Period Returns

**Simple Returns:** Use geometric compounding
$$R_{total} = (1 + R_1)(1 + R_2) \cdots (1 + R_n) - 1$$

**Log Returns:** Use simple addition
$$r_{total} = r_1 + r_2 + \cdots + r_n$$

This is why log returns are preferred for time series analysis!

## 3. Python Implementation

### 3.1 Calculating Simple Returns

Let's calculate simple returns using pandas:

### 3.2 Calculating Log Returns

Now let's calculate logarithmic returns:

In [ ]:
# Method 1: Using numpy log with price ratio
df_prices['Log_Return'] = np.log(df_prices['Price'] / df_prices['Price'].shift(1))

# Method 2: Using difference of log prices
df_prices['Log_Return_Alt'] = np.log(df_prices['Price']) - np.log(df_prices['Price'].shift(1))

# Method 3: Convert from simple returns
df_prices['Log_Return_From_Simple'] = np.log(1 + df_prices['Simple_Return'])

print("Log Return Calculation Methods")
print("=" * 60)
print("\nFirst 10 log returns:")
print(df_prices[['Price', 'Log_Return', 'Log_Return_Alt', 'Log_Return_From_Simple']].head(11))

# Verify all methods match
print("\nVerifying all methods produce identical results:")
print(f"  Method 1 vs Method 2: {(df_prices['Log_Return'] - df_prices['Log_Return_Alt']).abs().sum():.15f}")
print(f"  Method 1 vs Method 3: {(df_prices['Log_Return'] - df_prices['Log_Return_From_Simple']).abs().sum():.15f}")

# Log return statistics
print("\nLog Return Statistics:")
print(df_prices['Log_Return'].describe())

# Clean up
df_prices.drop(['Log_Return_Alt', 'Log_Return_From_Simple'], axis=1, inplace=True)

In [ ]:
# Method 1: Using pandas pct_change() - Most common
df_prices['Simple_Return'] = df_prices['Price'].pct_change()

# Verify calculation
print("Simple Return Calculation")
print("=" * 60)
print("\nFirst 10 returns:")
print(df_prices[['Price', 'Simple_Return']].head(11))

# Summary statistics
print("\nSimple Return Statistics:")
print(df_prices['Simple_Return'].describe())
print(f"\nDaily Mean: {df_prices['Simple_Return'].mean():.6f} ({df_prices['Simple_Return'].mean()*100:.4f}%)")
print(f"Annual Mean (252 days): {df_prices['Simple_Return'].mean() * 252:.6f} ({df_prices['Simple_Return'].mean()*252*100:.2f}%)")

## 4. Visualization and Analysis

## 5. Practical Applications

### 5.1 When to Use Simple vs Log Returns

**Use Simple Returns when:**
- ✅ Reporting to clients or stakeholders (more intuitive)
- ✅ Calculating portfolio returns from individual asset returns
- ✅ Cross-sectional analysis (comparing different assets at same time)
- ✅ Short holding periods with small returns

**Use Log Returns when:**
- ✅ Time series analysis and modeling
- ✅ Statistical analysis (closer to normal distribution)
- ✅ Multi-period aggregation (simple addition)
- ✅ Derivatives pricing and risk management
- ✅ Academic research

### 5.2 Common Pitfalls to Avoid

**Pitfall 1: Averaging Simple Returns**
```python
# WRONG: Don't average simple returns across time\navg_simple = df_prices['Simple_Return'].mean()  # ❌ Incorrect for time aggregation\n\n# RIGHT: Use geometric mean or log returns\ngeometric_mean = (1 + df_prices['Simple_Return']).prod() ** (1/len(df_prices)) - 1  # ✅\n# OR\navg_log = df_prices['Log_Return'].mean()  # ✅ Then convert if needed\n```\n\n**Pitfall 2: Mixing Return Types**\n- Don't add simple returns: (10% + 5% ≠ 15% total return)
- Don't multiply log returns: (log returns need to be summed, not multiplied)\n\n**Pitfall 3: Ignoring Compounding**\n- Over long periods, compounding effects become significant\n- Log returns naturally account for continuous compounding\n\n### 5.3 Real-World Example: Portfolio Performance

## Summary

This notebook provided a comprehensive introduction to return calculations in quantitative finance.

### Key Takeaways

✅ **Simple Returns** ($R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$)
- Intuitive and easy to understand
- Use for reporting and cross-sectional analysis
- Aggregate using geometric compounding: $(1+R_1)(1+R_2)\cdots(1+R_n) - 1$

✅ **Log Returns** ($r_t = \ln(\frac{P_t}{P_{t-1}})$)
- Time-additive: just sum them up!
- Symmetric and statistically convenient
- Preferred for time series analysis and modeling

✅ **When to Use Each**
- Simple returns → Client reporting, portfolio calculations, short periods
- Log returns → Time series modeling, derivatives pricing, academic research

✅ **Practical Skills Learned**
- Calculate both types of returns using pandas
- Visualize return distributions and identify normality
- Compute cumulative returns and wealth indices
- Analyze multi-asset portfolio performance
- Avoid common pitfalls (don't add simple returns across time!)\n\n### Common Pitfalls to Avoid\n\n❌ Don't add simple returns across time periods
❌ Don't forget to annualize returns when comparing\n❌ Don't mix return types in the same calculation
❌ Don't ignore compounding effects over long periods\n\n### Next Steps\n\n1. **Risk Metrics** - Learn about volatility, Sharpe ratio, and Value at Risk\n2. **Portfolio Theory** - Explore mean-variance optimization\n3. **Time Series Analysis** - ARIMA, GARCH models for returns\n4. **Real Data** - Apply these concepts to actual market data\n\n### Additional Resources\n\n**Books:**\n1. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson. (Chapters 1-3)\n2. Tsay, R.S. (2010). *Analysis of Financial Time Series*. Wiley. (Chapter 1)\n3. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. (Chapters 5-6)\n\n**Online:**\n- [QuantEdX Return Analysis Course](https://www.quantedx.com/courses/returns)\n- [Khan Academy: Returns and Compounding](https://www.khanacademy.org/economics-finance-domain)\n- [Investopedia: Return Calculations](https://www.investopedia.com/terms/r/return.asp)\n\n**Python Libraries:**\n- `pandas` - For return calculations\n- `numpy` - For mathematical operations\n- `empyrical` - For performance metrics\n- `pyfolio` - For portfolio analysis\n\n### Practice Exercises\n\nTry these on your own:\n\n1. Download real stock data and calculate both return types\n2. Compare the normality of simple vs log returns for different assets\n3. Build a portfolio and track its cumulative return over time\n4. Calculate rolling Sharpe ratios to assess risk-adjusted performance\n5. Analyze the impact of compounding frequency on total returns\n\n---\n\n**Estimated Completion Time**: 90-120 minutes\n\n**Difficulty Level**: Beginner\n\n**Prerequisites**: Basic Python, Basic Finance Concepts\n\n**Next Recommended Notebook**: `risk-metrics-intro.ipynb`\n\n---\n\n*Happy Learning! 🚀*

### 4.2 Cumulative Returns

Understanding how returns compound over time is crucial for portfolio analysis:

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(3, 2, figsize=(15, 12))

# 1. Return time series comparison
axes[0, 0].plot(df_prices.index, df_prices['Simple_Return'], alpha=0.7, label='Simple Return', linewidth=1.5)
axes[0, 0].plot(df_prices.index, df_prices['Log_Return'], alpha=0.7, label='Log Return', linewidth=1.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--', alpha=0.3)
axes[0, 0].set_title('Simple vs Log Returns Over Time', fontweight='bold', fontsize=12)
axes[0, 0].set_ylabel('Return')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Scatter plot: Simple vs Log returns
axes[0, 1].scatter(df_prices['Simple_Return'], df_prices['Log_Return'], alpha=0.5, s=20)
axes[0, 1].plot([-0.1, 0.1], [-0.1, 0.1], 'r--', linewidth=2, label='y=x line')
axes[0, 1].set_xlabel('Simple Return')
axes[0, 1].set_ylabel('Log Return')
axes[0, 1].set_title('Simple vs Log Returns Relationship', fontweight='bold', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Distribution of simple returns
axes[1, 0].hist(df_prices['Simple_Return'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1, 0].axvline(df_prices['Simple_Return'].mean(), color='red', linestyle='--', linewidth=2, 
                   label=f"Mean: {df_prices['Simple_Return'].mean():.4f}")
axes[1, 0].axvline(df_prices['Simple_Return'].median(), color='green', linestyle='--', linewidth=2,
                   label=f"Median: {df_prices['Simple_Return'].median():.4f}")
axes[1, 0].set_xlabel('Simple Return')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Simple Return Distribution', fontweight='bold', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Distribution of log returns
axes[1, 1].hist(df_prices['Log_Return'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].axvline(df_prices['Log_Return'].mean(), color='red', linestyle='--', linewidth=2,
                   label=f"Mean: {df_prices['Log_Return'].mean():.4f}")
axes[1, 1].axvline(df_prices['Log_Return'].median(), color='green', linestyle='--', linewidth=2,
                   label=f"Median: {df_prices['Log_Return'].median():.4f}")
axes[1, 1].set_xlabel('Log Return')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Log Return Distribution', fontweight='bold', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 5. Q-Q plot for simple returns
stats.probplot(df_prices['Simple_Return'].dropna(), dist="norm", plot=axes[2, 0])
axes[2, 0].set_title('Q-Q Plot: Simple Returns vs Normal', fontweight='bold', fontsize=12)
axes[2, 0].grid(True, alpha=0.3)

# 6. Q-Q plot for log returns
stats.probplot(df_prices['Log_Return'].dropna(), dist="norm", plot=axes[2, 1])
axes[2, 1].set_title('Q-Q Plot: Log Returns vs Normal', fontweight='bold', fontsize=12)
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print comparison statistics
print("\nReturn Statistics Comparison")
print("=" * 60)
print(f"{'Metric':<25} {'Simple Return':>15} {'Log Return':>15}")
print("-" * 60)
print(f"{'Mean':<25} {df_prices['Simple_Return'].mean():>15.6f} {df_prices['Log_Return'].mean():>15.6f}")
print(f"{'Std Dev':<25} {df_prices['Simple_Return'].std():>15.6f} {df_prices['Log_Return'].std():>15.6f}")
print(f"{'Min':<25} {df_prices['Simple_Return'].min():>15.6f} {df_prices['Log_Return'].min():>15.6f}")
print(f"{'Max':<25} {df_prices['Simple_Return'].max():>15.6f} {df_prices['Log_Return'].max():>15.6f}")
print(f"{'Skewness':<25} {df_prices['Simple_Return'].skew():>15.6f} {df_prices['Log_Return'].skew():>15.6f}")
print(f"{'Kurtosis':<25} {df_prices['Simple_Return'].kurtosis():>15.6f} {df_prices['Log_Return'].kurtosis():>15.6f}")

### 4.1 Comparing Simple and Log Returns

Let's visualize the difference between simple and log returns:

## 5. Practical Applications

## Summary

This notebook covered return calculations 101. Key takeaways:

- Practical implementation with Python
- Visualizations and interpretations
- Real-world applications
- Best practices and pitfalls

### Next Steps

- Practice with real market data
- Explore related topics
- Build your own variations

### Additional Resources

- [QuantEdX Courses](https://www.quantedx.com/courses)
- [Community Forum](https://www.quantedx.com/community)
- [GitHub Repository](https://github.com/quantedx)